___
# <center>Matriz de confusão, medidas de erro e ponto de corte</center>
___

## Aula 21

**Objetivo da aula:** ao final desta aula, você deve ser capaz de:

 * transformar a probabilidade prevista por uma regressão logística em decisão,
   escolhendo um ponto de corte;
 * montar a matriz de confusão e dizer, no vocabulário do caso, o que é cada
   acerto e cada erro;
 * calcular acurácia, precisão, sensibilidade, especificidade e F1;
 * comparar o modelo com o modelo trivial quando o evento é raro;
 * escolher o corte pela capacidade e pelo erro mais grave.

A aula é feita nos slides e no Projeto 06. Este notebook repete as contas em
Python, para estudar em casa e para a lista 6.

___
<div id="indice"></div>

## Índice

- [A base e o modelo da aula 20](#base)
- [1. Probabilidade prevista](#prob)
- [2. Ponto de corte e matriz de confusão](#matriz)
- [3. Medidas de erro](#medidas)
- [4. O corte muda as medidas](#cortes)
- [5. Exercício: condenações altas, um evento raro](#exercicio)

___
<div id="base"></div>

# A base e o modelo da aula 20

O banco quer **oferecer acordo antes da sentença** nos processos que tende a
perder. Para isso, usa a regressão logística da aula 20, que estima a
probabilidade de sentença favorável ao autor.

**Atenção: a base é simulada.** Ela imita a carteira de ações revisionais de
financiamento de veículo que vimos com o BTG, mas cada linha foi sorteada no
computador. É a mesma base da aula 20, pública, lida direto da internet. Os
resultados ilustram os métodos e **não dizem nada sobre processos reais**.

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from plotnine import *

processos = pd.read_csv("https://raw.githubusercontent.com/jtrecenti/202662-cdad2/main/dados/processos_simulados.csv")

formula = "sentenca_favoravel ~ taxa_juros + escritorio_massa + juizado + idade_autor"
logit2 = smf.logit(formula, data=processos).fit()
np.exp(logit2.params).round(3)   # razões de chances, como na aula 20

[Volta ao Índice](#indice)

___
<div id="prob"></div>

# 1. Probabilidade prevista

`predict()` devolve, para cada processo, a probabilidade prevista de sentença
favorável ao autor. Guardamos numa coluna nova.

In [ ]:
processos["prob"] = logit2.predict(processos)
processos[["taxa_juros", "escritorio_massa", "juizado", "idade_autor", "prob", "sentenca_favoravel"]].head()

O gráfico separa os processos pelo que aconteceu de fato. Os favoráveis ao
autor ficam mais à direita, mas as duas distribuições se sobrepõem: nenhum
corte separa os grupos sem erro.

In [ ]:
(ggplot(processos, aes(x="prob", fill="factor(sentenca_favoravel)"))
 + geom_histogram(binwidth=0.05, alpha=0.6, position="identity")
 + labs(x="probabilidade prevista de sentença favorável", y="processos",
        fill="sentença favorável")
 + theme_minimal())

[Volta ao Índice](#indice)

___
<div id="matriz"></div>

# 2. Ponto de corte e matriz de confusão

Com corte de 0,5, o banco oferece acordo nos processos com probabilidade igual
ou acima de 0,5. A matriz de confusão cruza a **decisão** com o **que
aconteceu**.

In [ ]:
corte = 0.5
processos["oferece"] = (processos["prob"] >= ____).astype(int)

pd.crosstab(processos["oferece"], processos["sentenca_favoravel"],
            rownames=["oferece acordo"], colnames=["sentença favorável"])

Os quatro números, com nome:

| | sentença favorável (1) | desfavorável (0) |
|---|---|---|
| **oferece (1)** | VP: ofereceu e perderia | FP: ofereceu e ganharia |
| **não oferece (0)** | FN: não ofereceu e perdeu | VN: não ofereceu e ganhou |

In [ ]:
y = processos["sentenca_favoravel"]
decisao = processos["oferece"]

vp = ((decisao == 1) & (y == 1)).sum()
fp = ((decisao == 1) & (y == ____)).sum()
fn = ((decisao == ____) & (y == 1)).sum()
vn = ((decisao == 0) & (y == 0)).sum()
vp, fp, fn, vn

[Volta ao Índice](#indice)

___
<div id="medidas"></div>

# 3. Medidas de erro

| medida | conta | pergunta que responde |
|---|---|---|
| acurácia | (VP + VN) / total | que fração das decisões acertou? |
| precisão | VP / (VP + FP) | dos acordos oferecidos, quantos eram necessários? |
| sensibilidade (revocação) | VP / (VP + FN) | dos processos que o banco perderia, em quantos ofereceu? |
| especificidade | VN / (VN + FP) | dos que o banco ganharia, em quantos não ofereceu? |
| F1 | 2 × precisão × sensibilidade / (precisão + sensibilidade) | as duas num número só |

In [ ]:
total = vp + fp + fn + vn
acuracia = (vp + vn) / total
precisao = vp / (vp + ____)
sensibilidade = vp / (vp + ____)
especificidade = vn / (vn + fp)
f1 = 2 * precisao * sensibilidade / (precisao + sensibilidade)

pd.Series({"acurácia": acuracia, "precisão": precisao, "sensibilidade": sensibilidade,
           "especificidade": especificidade, "F1": f1}).round(3)

O `scikit-learn` faz as mesmas contas, e vamos usá-lo a partir da aula 23:

In [ ]:
from sklearn.metrics import confusion_matrix, precision_score, recall_score

print(confusion_matrix(y, decisao))   # linhas: o que aconteceu; colunas: a decisão
precision_score(y, decisao), recall_score(y, decisao)

**Pergunta.** Traduza a precisão e a sensibilidade em uma frase cada, para a
diretoria jurídica do banco.

*(responda aqui)*

[Volta ao Índice](#indice)

___
<div id="cortes"></div>

# 4. O corte muda as medidas

A função abaixo faz as contas da parte 3 para qualquer corte. Com ela, montamos
uma tabela com vários cortes.

In [ ]:
def medidas(y, prob, corte):
    decisao = prob >= corte
    vp = (decisao & (y == 1)).sum()
    fp = (decisao & (y == 0)).sum()
    fn = (~decisao & (y == 1)).sum()
    vn = (~decisao & (y == 0)).sum()
    return {"corte": corte, "ofertas": vp + fp, "VP": vp, "FP": fp, "FN": fn, "VN": vn,
            "acurácia": (vp + vn) / len(y),
            "precisão": vp / (vp + fp) if vp + fp else np.nan,
            "sensibilidade": vp / (vp + fn)}

tabela = pd.DataFrame([medidas(processos["sentenca_favoravel"], processos["prob"], c)
                       for c in [0.3, 0.4, 0.5, 0.6, 0.68, 0.7, 0.8]])
tabela.round(3)

**Perguntas.**

1. O que acontece com a precisão e a sensibilidade quando o corte sobe? Por quê?
2. O jurídico do banco consegue negociar **300 acordos**. Qual o menor corte da
   tabela que cabe? Quantos processos perdidos ficam sem oferta nesse corte?

*(responda aqui)*

[Volta ao Índice](#indice)

___
<div id="exercicio"></div>

# 5. Exercício: condenações altas, um evento raro

O banco também quer sinalizar, para um advogado sênior, os processos que vão
terminar com **condenação de R\$ 20 mil ou mais**. Eles são poucos.

In [ ]:
processos["condenacao_alta"] = (processos["valor_condenacao"].fillna(0) >= 20).astype(int)
processos["condenacao_alta"].mean()     # a fração de condenações altas

modelo_alta = smf.logit("condenacao_alta ~ valor_causa + taxa_juros", data=processos).fit()
processos["prob_alta"] = modelo_alta.predict(processos)

# a) o modelo trivial nunca sinaliza nada: qual a acurácia dele?
acuracia_trivial = 1 - processos["condenacao_alta"].____()
acuracia_trivial

In [ ]:
# b) as medidas do modelo com corte 0,5 e com corte 0,1
pd.DataFrame([medidas(processos["condenacao_alta"], processos["prob_alta"], c)
              for c in [____, ____]]).round(3)

**Perguntas.**

1. Compare a acurácia do modelo trivial com a do modelo nos dois cortes. Qual é
   maior?
2. Qual dos três você usaria para avisar o advogado sênior? Use a sensibilidade
   e a precisão.
3. O que a acurácia esconde aqui?

*(responda aqui)*

[Volta ao Índice](#indice)